# ReviveOS — sessão sem login

Notebook para exploração interativa leve, sem cadastro. Abra-o numa sessão Binder e execute as células na ordem. Não é um desktop Windows nem executa AutoCAD.

Use somente exemplos ou dados que possam ser públicos. Não envie projetos confidenciais, credenciais ou informações pessoais. A sessão pode encerrar e seus arquivos não persistem; baixe os resultados antes de sair. Não deixe tarefas contínuas nem tente impedir a expiração.


In [ ]:
import json, math, hashlib, platform, uuid
from pathlib import Path
from collections import Counter
resultados = {}
print("Sistema da sessão:", platform.system())
print("Nenhuma aceleração de hardware local é presumida.")


## Cálculo de exemplo
Altere os valores conforme seu exercício. Este exemplo calcula áreas e não modifica desenhos.


In [ ]:
raio = 10.0
if not math.isfinite(raio) or raio < 0:
    raise ValueError("Raio deve ser finito e não negativo")
resultados["circulo"] = {"raio": raio, "area": math.pi * raio**2, "perimetro": 2 * math.pi * raio}
print(json.dumps(resultados["circulo"], indent=2))


## CSV opcional
Para testar, envie um arquivo não confidencial pelo botão Upload do Jupyter com o nome `entrada.csv`. A célula apenas resume linhas e colunas; o limite de entrada é 8 MiB.


In [ ]:
import csv
p = Path("entrada.csv")
if p.exists():
    if p.stat().st_size > 8 * 1024 * 1024:
        raise ValueError("CSV excede o limite de 8 MiB deste exemplo")
    with p.open(encoding="utf-8-sig", newline="") as stream:
        rows = csv.reader(stream)
        header = next(rows, [])
        count = sum(1 for _ in rows)
    resultados["csv"] = {"colunas": header, "linhas_sem_cabecalho": count, "sha256": hashlib.sha256(p.read_bytes()).hexdigest()}
    print(json.dumps(resultados["csv"], indent=2))
else:
    print("Sem entrada.csv: etapa opcional ignorada, nenhum dado inventado.")


## DXF ASCII opcional
Envie um DXF de exemplo não confidencial com o nome `entrada.dxf`. Conta os tipos de entidades na seção ENTITIES, sem editar o arquivo. Não interpreta DWG, blocos expandidos ou desempenho do AutoCAD.


In [ ]:
p = Path("entrada.dxf")
if p.exists():
    if p.stat().st_size > 8 * 1024 * 1024:
        raise ValueError("DXF excede o limite de 8 MiB deste exemplo")
    data = p.read_bytes()
    if data.startswith(b"AutoCAD Binary DXF"):
        raise ValueError("Este exemplo aceita apenas DXF ASCII")
    lines = data.decode("latin-1").splitlines()
    if len(lines) % 2:
        raise ValueError("DXF incompleto: pares código/valor inválidos")
    pairs = [(int(lines[i].strip()), lines[i+1].strip()) for i in range(0, len(lines), 2)]
    inside = False
    counts = Counter()
    for i, (group, value) in enumerate(pairs):
        if group == 0 and value == "SECTION":
            inside = i+1 < len(pairs) and pairs[i+1] == (2, "ENTITIES")
        elif group == 0 and value in ("ENDSEC", "EOF"):
            inside = False
        elif inside and group == 0:
            counts[value] += 1
    resultados["dxf"] = {"entidades_por_tipo": dict(sorted(counts.items())), "sha256": hashlib.sha256(data).hexdigest()}
    print(json.dumps(resultados["dxf"], indent=2))
else:
    print("Sem entrada.dxf: etapa opcional ignorada.")


## Baixar o resultado
A célula cria um arquivo novo com SHA-256. Clique nos links para baixar o JSON e o checksum no HD do notebook. Não confunda arquivos na sessão com arquivos já salvos localmente.


In [ ]:
text = json.dumps(resultados, indent=2, ensure_ascii=False).encode("utf-8")
name = "reviveos-resultado-" + uuid.uuid4().hex + ".json"
with open(name, "xb") as stream:
    stream.write(text)
checksum = hashlib.sha256(text).hexdigest()
with open(name + ".sha256", "x", encoding="utf-8") as stream:
    stream.write(checksum + "  " + name + "\n")
print("SHA-256:", checksum)
try:
    from IPython.display import display, FileLink
    display(FileLink(name), FileLink(name + ".sha256"))
except ImportError:
    print("Arquivos gerados:", name, name + ".sha256")
